[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/15_mlp.ipynb)

# 🟠 Medium: SwiGLU MLP

Implement the **SwiGLU MLP** (feed-forward network) used in modern LLMs like LLaMA.

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\big)$$

where $\text{SiLU}(x) = x \cdot \sigma(x)$

### Signature
```python
class SwiGLUMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.gate_proj`: `nn.Linear(d_model, d_ff)`
- `self.up_proj`: `nn.Linear(d_model, d_ff)`
- `self.down_proj`: `nn.Linear(d_ff, d_model)`
- Activation: **SiLU** (a.k.a. Swish) — `F.silu` or implement as `x * torch.sigmoid(x)`

### Why SwiGLU?
Unlike the classic `Linear → ReLU/GELU → Linear` FFN, SwiGLU uses a **gating mechanism**:
the gate projection controls information flow, while the up projection provides the content.
This consistently outperforms standard FFNs in practice (PaLM, LLaMA, Mistral all use it).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff)
        self.up_proj = nn.Linear(d_model, d_ff)
        self.down_proj = nn.Linear(d_ff, d_model)

    def forward(self, x):

        # 수식: down_proj(SiLU(gate_proj(x)) * up_proj(x))
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

In [4]:
# 🧪 Debug
mlp = SwiGLUMLP(d_model=64, d_ff=128)
x = torch.randn(2, 8, 64)
out = mlp(x)
print("Output shape:", out.shape)  # (2, 8, 64)
print("Params:", sum(p.numel() for p in mlp.parameters()))

Output shape: torch.Size([2, 8, 64])
Params: 24896


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('mlp')


🧪 Testing: SwiGLU MLP (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Parameter shapes (1.5ms)
  ✅ [2/5] Forward output shape (1.1ms)
  ✅ [3/5] Numerical correctness (48.8ms)
  ✅ [4/5] 2-D input (1.0ms)
  ✅ [5/5] Gradient flow (25.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (77.9ms total)
  Progress saved. Run status() to see your dashboard.



In [6]:
mlp

SwiGLUMLP(
  (gate_proj): Linear(in_features=64, out_features=128, bias=True)
  (up_proj): Linear(in_features=64, out_features=128, bias=True)
  (down_proj): Linear(in_features=128, out_features=64, bias=True)
)

In [16]:
for name, p in mlp.named_parameters():
    print(name, p.shape, p.numel(), p, sep='\n')
    print()

gate_proj.weight
torch.Size([128, 64])
8192
Parameter containing:
tensor([[ 0.0526, -0.0204,  0.0336,  ..., -0.0156,  0.0066, -0.1129],
        [ 0.0777, -0.0547,  0.1226,  ...,  0.0342, -0.1076,  0.0009],
        [ 0.0040, -0.1072, -0.0670,  ..., -0.1075, -0.0454,  0.0767],
        ...,
        [-0.0105, -0.1003, -0.1146,  ...,  0.0272,  0.0871,  0.0044],
        [ 0.0619,  0.0333,  0.1057,  ...,  0.0833,  0.0759, -0.0263],
        [-0.0325, -0.0719, -0.0406,  ...,  0.0821, -0.0425, -0.1031]],
       requires_grad=True)

gate_proj.bias
torch.Size([128])
128
Parameter containing:
tensor([ 0.0482,  0.1173, -0.0351, -0.0742, -0.0817, -0.0651, -0.0731, -0.0054,
        -0.1205, -0.1236, -0.0081,  0.1138,  0.0300, -0.0755,  0.1243, -0.0291,
         0.0904,  0.0547,  0.1172, -0.1040,  0.0747,  0.0394, -0.1102,  0.0131,
        -0.0186,  0.1028, -0.0864,  0.1170,  0.0273,  0.1006,  0.1169,  0.1147,
        -0.1148, -0.0266,  0.0993,  0.0864, -0.0517, -0.0575, -0.0544, -0.0778,
         0.01